# Step 0: Environment Initialization 

In [40]:
# Basic imports and verify torch is cuda compatible (2.5.1+cu121)
import os
import torch
from torchvision import datasets, transforms
from torch import nn
from tqdm.auto import tqdm

torch.__version__

'2.5.1+cu121'

In [41]:
# Verify we are working on GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

# Step 1: Load & Process Data

In [42]:
# Define transformation function, TODO TSFR FUCNTION IS CURRENTLY MOST BASIC POSSIBLE VERSION, REVISE LATER AS IT AFFECTS MODEL PERFORMANCE
transform = transforms.ToTensor()

# Load data #TODO data is currently toy data: mnist dataset of handwritten digits
mnist_train_data = datasets.MNIST(root="./mnist_data", transform=transform, train=True, download=False)
mnist_test_data = datasets.MNIST(root="./mnist_data", transform=transform, train=False, download=False)

# Create DataLoaders
mnist_train_dataloader = torch.utils.data.DataLoader(mnist_train_data, batch_size=32, shuffle=True)
mnist_test_dataloader = torch.utils.data.DataLoader(mnist_test_data, batch_size=32, shuffle=True)

# Step once to test
for images, labels in mnist_train_dataloader:
    print(images.shape, labels.shape)
    break  # Just checking one batch

torch.Size([32, 1, 28, 28]) torch.Size([32])


# Step 2: Initialize Model Architecture

In [43]:
#TODO Model architecture is currently just a copy of a CNN
#TODO CustomModel currently has a few hard-coded features to be compatible with MNIST data, will need to be adjusted to accept BRATS data
class CustomModel(nn.Module):
  def __init__(self, input_shape, hidden_units, output_shape):
      super().__init__()
      
      # First convolutional block
      self.conv_1 = nn.Sequential(
          nn.Conv2d(in_channels=input_shape,
                    out_channels=hidden_units,
                    kernel_size=3,
                    stride=1,
                    padding=1),
          nn.ReLU(),
          nn.Conv2d(in_channels=hidden_units,
                    out_channels=hidden_units,
                    kernel_size=3,
                    stride=1,
                    padding=1),
          nn.ReLU(),
          nn.MaxPool2d(kernel_size=2, stride=2)  # Halves the spatial dimensions
      )
      
      # Second convolutional block
      self.conv_2 = nn.Sequential(
          nn.Conv2d(in_channels=hidden_units,
                    out_channels=hidden_units,
                    kernel_size=3,
                    stride=1,
                    padding=1),
          nn.ReLU(),
          nn.Conv2d(in_channels=hidden_units,
                    out_channels=hidden_units,
                    kernel_size=3,
                    stride=1,
                    padding=1),
          nn.ReLU(),
          nn.MaxPool2d(kernel_size=2, stride=2)  # Halves the spatial dimensions again
      )
      
      # Compute the number of flattened features dynamically
      self.flattened_size = self._get_flattened_size(input_shape, hidden_units)

      # Fully connected layer (classifier)
      self.classifier = nn.Sequential(
          nn.Flatten(),
          nn.Linear(in_features=self.flattened_size,  # Fix here
                    out_features=output_shape)
      )

  def _get_flattened_size(self, input_shape, hidden_units):
      """ Pass a dummy tensor through the conv layers to get the output size. """
      with torch.no_grad():
          dummy_input = torch.randn(1, input_shape, 28, 28)  # MNIST images are 28x28
          output = self.conv_2(self.conv_1(dummy_input))
          return output.view(1, -1).shape[1]  # Flattened size
          
  def forward(self, x):
    return self.classifier(self.conv_2(self.conv_1(x)))

# Step 3: Define Train and Test Step Behavior

In [44]:
def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer):
  model.train() # Convert model to train mode

  train_loss, train_acc = 0, 0
  for batch, (X, y) in enumerate(dataloader):
    X, y = X.to(device), y.to(device)
    # Do the forward pass
    y_pred = model(X)
    # Calculate th loss
    loss = loss_fn(y_pred, y)
    train_loss += loss.item()
    # Optimizer zero grad
    optimizer.zero_grad()
    # Loss backward
    loss.backward()
    # Optimizer step
    optimizer.step()

    # Calc accuracy
    y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
    train_acc += (y_pred_class==y).sum().item()/len(y_pred)

  train_loss = train_loss / len(dataloader)
  train_acc = train_acc /  len(dataloader)
  return train_loss, train_acc

In [45]:
def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module):
  model.eval() # Convert model to inference mode

  test_loss, test_acc = 0, 0

  with torch.inference_mode():
    for batch, (X, y) in enumerate(dataloader):
      X, y = X.to(device), y.to(device)
      # Forward pass
      test_pred_logits = model(X)
      # Calculate the loss
      loss = loss_fn(test_pred_logits, y)
      test_loss += loss.item()
      # Calc acc
      test_pred_labels = test_pred_logits.argmax(dim=1)
      test_acc += ((test_pred_labels==y)).sum().item()/len(test_pred_labels)

    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc

In [46]:
def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int):

    # Create empty results dictionary
    results = {"train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }

    # loop through training and testing steps EPOCH times
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer)
        test_loss, test_acc = test_step(model=model,
            dataloader=test_dataloader,
            loss_fn=loss_fn)

        print(
            f"Epoch: {epoch+1} | "
            f"train_loss: {train_loss:.3f} | "
            f"train_acc: {train_acc:.3f} | "
            f"test_loss: {test_loss:.3f} | "
            f"test_acc: {test_acc:.3f}"
        )

        # Update results dictionary
        results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
        results["train_acc"].append(train_acc.item() if isinstance(train_acc, torch.Tensor) else train_acc)
        results["test_loss"].append(test_loss.item() if isinstance(test_loss, torch.Tensor) else test_loss)
        results["test_acc"].append(test_acc.item() if isinstance(test_acc, torch.Tensor) else test_acc)

    return results

# Step 4: Training the Model

In [ ]:
# Init an instance of custom model
model = CustomModel(input_shape=1, # All 3 of the inputs to the custom model function must be tuned to the character of data it is being fitted to
                  hidden_units=16,
                  output_shape=10).to(device)

# Set up loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=0.1)

In [48]:
from timeit import default_timer as timer
start_time = timer()

# Train loop
NUM_EPOCHS = 5
model_results = train(model=model,
                        train_dataloader=mnist_train_dataloader,
                        test_dataloader=mnist_train_dataloader,
                        optimizer=optimizer,
                        loss_fn=loss_fn,
                        epochs=NUM_EPOCHS)

end_time = timer()
print(f"Training time was {end_time-start_time} seconds.")

 20%|██        | 1/5 [00:08<00:34,  8.63s/it]

Epoch: 1 | train_loss: 0.200 | train_acc: 0.936 | test_loss: 0.067 | test_acc: 0.980


 40%|████      | 2/5 [00:17<00:26,  8.69s/it]

Epoch: 2 | train_loss: 0.063 | train_acc: 0.980 | test_loss: 0.066 | test_acc: 0.979


 60%|██████    | 3/5 [00:25<00:17,  8.53s/it]

Epoch: 3 | train_loss: 0.049 | train_acc: 0.984 | test_loss: 0.037 | test_acc: 0.988


 80%|████████  | 4/5 [00:36<00:09,  9.45s/it]

Epoch: 4 | train_loss: 0.040 | train_acc: 0.987 | test_loss: 0.032 | test_acc: 0.990


100%|██████████| 5/5 [00:44<00:00,  8.92s/it]

Epoch: 5 | train_loss: 0.034 | train_acc: 0.990 | test_loss: 0.027 | test_acc: 0.992
Training time was 44.57911510008853 seconds.


In [ ]:
# NOTE There is currently no code which saves/exports the model, "model" and the "model_results" dictionary does not exist outside of RAM currently.